In [ ]:
!pip install torch transformers pandas scikit-learn sentencepiece protobuf peft

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from google.colab import drive
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report

# 1. Mount Storage and Target GPU
print("[*] Mounting Google Drive...")
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Execution Target: {device}")

# 2. Define Architecture Paths
BASE_DIR = "/content/drive/MyDrive/processed"
MODEL_NAME = "microsoft/deberta-v3-small"

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(f"CRITICAL: '{BASE_DIR}' not found. Ensure Drive is mounted and folders match.")

print("[*] Initializing shared DeBERTa tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

[*] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[+] Execution Target: cuda
[*] Initializing shared DeBERTa tokenizer...


In [ ]:
print("\n[*] Training: Cross-Encoder (Indirect Context - BIPIA)")

class CrossEncoderDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=384):
        df = pd.read_csv(csv_path)
        self.labels = df["label"].values
        # Paired input: [intent] paired_text [SEP] [context] text
        self.encodings = tokenizer(
            df["paired_text"].fillna("").astype(str).tolist(),
            df["text"].fillna("").astype(str).tolist(),
            padding="max_length", truncation=True, max_length=max_length, return_tensors="pt"
        )
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# 1. Load Data
train_loader_ce = DataLoader(CrossEncoderDataset(os.path.join(BASE_DIR, "indirect_context/train.csv"), tokenizer), batch_size=32, shuffle=True)
test_loader_ce = DataLoader(CrossEncoderDataset(os.path.join(BASE_DIR, "indirect_context/test.csv"), tokenizer), batch_size=64, shuffle=False)

# 2. Build Model (Explicitly locking base weights to FP32 to prevent downcast leaks)
model_ce = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, torch_dtype=torch.float32).to(device)
optimizer_ce = AdamW(model_ce.parameters(), lr=2e-5, weight_decay=0.01, eps=1e-6)

# FIXED: Modern PyTorch 2.x syntax for GradScaler
scaler_ce = torch.amp.GradScaler('cuda')

total_steps_ce = len(train_loader_ce) * 3
scheduler_ce = get_linear_schedule_with_warmup(optimizer_ce, num_warmup_steps=int(total_steps_ce * 0.1), num_training_steps=total_steps_ce)

# 3. Training Loop
for epoch in range(3):
    model_ce.train()
    total_loss = 0
    for batch_idx, batch in enumerate(train_loader_ce):
        optimizer_ce.zero_grad()
        input_ids, attention_mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            loss = model_ce(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss

        scaler_ce.scale(loss).backward()

        # --- PYTORCH AMP BUG FIX ---
        # Intercept rogue FP16 gradients from DeBERTa's attention block and cast them back to FP32
        for param in model_ce.parameters():
            if param.grad is not None and param.grad.dtype == torch.float16:
                param.grad = param.grad.to(torch.float32)
        # ---------------------------

        scaler_ce.unscale_(optimizer_ce)
        torch.nn.utils.clip_grad_norm_(model_ce.parameters(), max_norm=1.0)
        scaler_ce.step(optimizer_ce)
        scaler_ce.update()
        scheduler_ce.step()
        total_loss += loss.item()
    print(f"[+] Epoch {epoch+1}/3 CE Loss: {total_loss / len(train_loader_ce):.4f}")

# 4. Evaluate & Save
model_ce.eval()
preds_ce, true_ce = [], []
with torch.no_grad():
    for batch in test_loader_ce:
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model_ce(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
        preds_ce.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        true_ce.extend(batch["labels"].numpy())

print("\n" + "="*60 + "\n          CROSS-ENCODER REPORT (BIPIA)\n" + "="*60)
print(classification_report(true_ce, preds_ce, target_names=["Benign", "Malicious"]))

ce_save = os.path.join(BASE_DIR, "cross_encoder_weights")
os.makedirs(ce_save, exist_ok=True)
model_ce.save_pretrained(ce_save)
tokenizer.save_pretrained(ce_save)
print(f"[+] Saved Cross-Encoder to {ce_save}")

del model_ce, optimizer_ce, scaler_ce
torch.cuda.empty_cache()


[*] Training: Cross-Encoder (Indirect Context - BIPIA)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.de

[+] Epoch 1/3 CE Loss: 0.2021
[+] Epoch 2/3 CE Loss: 0.0896
[+] Epoch 3/3 CE Loss: 0.0707

          CROSS-ENCODER REPORT (BIPIA)
              precision    recall  f1-score   support

      Benign       0.95      0.98      0.97      7000
   Malicious       0.98      0.95      0.97      7000

    accuracy                           0.97     14000
   macro avg       0.97      0.97      0.97     14000
weighted avg       0.97      0.97      0.97     14000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[+] Saved Cross-Encoder to /content/drive/MyDrive/processed/cross_encoder_weights


In [ ]:
!pip install --upgrade torchao peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
print("\n[*] Training: DeBERTa-LoRA (Repo File - ProdNull)")

class SingleTextDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_length=256):
        df = pd.read_csv(csv_path)
        self.labels = df["label"].values
        self.encodings = tokenizer(
            df["text"].fillna("").astype(str).tolist(),
            padding="max_length", truncation=True, max_length=max_length, return_tensors="pt"
        )
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# 1. Load Data (ONLY repo_file)
train_loader_lora = DataLoader(SingleTextDataset(os.path.join(BASE_DIR, "repo_file/train.csv"), tokenizer), batch_size=32, shuffle=True)
test_loader_lora = DataLoader(SingleTextDataset(os.path.join(BASE_DIR, "repo_file/test.csv"), tokenizer), batch_size=64, shuffle=False)

# 2. Build LoRA Architecture (Explicitly locking base weights to FP32)
base_model_lora = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, torch_dtype=torch.float32)
model_lora = get_peft_model(base_model_lora, LoraConfig(task_type=TaskType.SEQ_CLS, r=8, target_modules=["query_proj", "value_proj"])).to(device)
optimizer_lora = AdamW(model_lora.parameters(), lr=2e-4, eps=1e-6)

# FIXED: Modern PyTorch 2.x syntax for GradScaler
scaler_lora = torch.amp.GradScaler('cuda')

total_steps_lora = len(train_loader_lora) * 3
scheduler_lora = get_linear_schedule_with_warmup(optimizer_lora, num_warmup_steps=int(total_steps_lora * 0.1), num_training_steps=total_steps_lora)

# 3. Training Loop
for epoch in range(3):
    model_lora.train()
    total_loss = 0
    for batch in train_loader_lora:
        optimizer_lora.zero_grad()
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            loss = model_lora(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device), labels=batch["labels"].to(device)).loss

        scaler_lora.scale(loss).backward()

        # --- PYTORCH AMP BUG FIX ---
        for param in model_lora.parameters():
            if param.grad is not None and param.grad.dtype == torch.float16:
                param.grad = param.grad.to(torch.float32)
        # ---------------------------

        scaler_lora.unscale_(optimizer_lora)
        torch.nn.utils.clip_grad_norm_(model_lora.parameters(), max_norm=1.0)
        scaler_lora.step(optimizer_lora)
        scaler_lora.update()
        scheduler_lora.step()
        total_loss += loss.item()
    print(f"[+] Epoch {epoch+1}/3 LoRA Loss: {total_loss / len(train_loader_lora):.4f}")

# 4. Evaluate & Save
model_lora.eval()
preds_lora, true_lora = [], []
with torch.no_grad():
    for batch in test_loader_lora:
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model_lora(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
        preds_lora.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        true_lora.extend(batch["labels"].numpy())

print("\n" + "="*60 + "\n          DEBERTA-LORA REPORT (ProdNull)\n" + "="*60)
print(classification_report(true_lora, preds_lora, target_names=["Benign", "Malicious"]))

lora_save = os.path.join(BASE_DIR, "deberta_lora_weights")
os.makedirs(lora_save, exist_ok=True)
model_lora.save_pretrained(lora_save)
print(f"[+] Saved LoRA Adapters to {lora_save}")

del model_lora, optimizer_lora, scaler_lora
torch.cuda.empty_cache()


[*] Training: DeBERTa-LoRA (Repo File - ProdNull)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.de

[+] Epoch 1/3 LoRA Loss: 0.6947
[+] Epoch 2/3 LoRA Loss: 0.5920
[+] Epoch 3/3 LoRA Loss: 0.4202

          DEBERTA-LORA REPORT (ProdNull)
              precision    recall  f1-score   support

      Benign       0.81      0.87      0.84       551
   Malicious       0.87      0.81      0.83       583

    accuracy                           0.84      1134
   macro avg       0.84      0.84      0.84      1134
weighted avg       0.84      0.84      0.84      1134

[+] Saved LoRA Adapters to /content/drive/MyDrive/processed/deberta_lora_weights
